# EP2 · TLY1102 — Red Neuronal Convolucional para clasificar personajes de Los Simpsons

**Evaluación Parcial N°2 — Técnicas Avanzadas de Machine Learning I**
Presentación y defensa técnica · Informe reproducible

---

## 1. Problema de negocio

Una plataforma de contenido audiovisual necesita **etiquetar automáticamente** fotogramas de una serie
animada para alimentar su buscador y sus recomendaciones. Hoy ese etiquetado es manual: una persona
mira el fotograma y escribe qué personaje aparece. El costo escala linealmente con el catálogo.

**Solución propuesta:** un clasificador de imágenes que, dado un fotograma, devuelva el personaje.

### KPIs

| KPI | Definición | Meta |
|---|---|---|
| **Accuracy en test** | proporción de fotogramas bien etiquetados sobre un conjunto no usado en el diseño | ≥ 0,90 |
| **F1 macro** | promedio no ponderado del F1 de los 18 personajes | ≥ 0,90 |
| **Costo de inferencia** | parámetros del modelo y tiempo de entrenamiento en CPU | informar y comparar |
| **Reducción de etiquetado manual** | 1 − (errores del modelo / trabajo manual) | ≥ 80 % del volumen |

### Objetivos

1. Preparar el conjunto de imágenes en formato tensorial apto para una CNN.
2. Diseñar, implementar y justificar una arquitectura convolucional.
3. Evaluar con accuracy, precision, recall, F1 y matriz de confusión.
4. **Comparar objetivamente** la CNN contra una red densa (MLP) sobre los mismos datos.
5. Interpretar los mapas de características y los mapas de activación por clase.
6. Analizar los errores y sus causas.

### Metodología

Se sigue **CRISP-DM**: comprensión del negocio (arriba), comprensión de los datos (§3),
preparación (§4), modelamiento (§5–§6), evaluación (§7–§10) y despliegue (el modelo exportado en
`models/`).

---

## 2. Continuidad con la evaluación anterior y **variante propia**

### Continuidad

Se reutiliza el **mismo conjunto de datos** de la evaluación anterior (Los Simpsons, 18 personajes),
tal como recomienda la pauta, para que la comparación entre arquitecturas sea sobre el mismo problema.

### Por qué hubo que cambiar el dataset respecto de la EP1

La pauta autoriza el cambio "siempre que se justifique técnicamente". La justificación es de
**compatibilidad de modalidad de dato**:

- El conjunto de la EP1 era **monocanal (escala de grises), 28×28 px**, con dígitos/letras aislados y
  fondo limpio.
- Este conjunto es **tricanal (RGB), resolución variable (256–1912 px), fondo complejo** y 18 clases.
- Una CNN diseñada para el primero no es comparable con una red densa entrenada sobre el segundo:
  cambiarían a la vez la arquitectura **y** la naturaleza del dato. La comparación exigida perdería
  validez.

Por eso se usa el conjunto tricanal RGB, que permite comparar **CNN vs MLP sobre el mismo tensor de
entrada**.

### Variante propia (exigida por la pauta)

> **Tope de 800 imágenes por clase + partición propia con semilla 2026.**

Dos efectos, ambos medibles:

1. **Reduce el desbalanceo** de 6,27× (Homer 2.246 vs Nelson 358) a ≈ 2,2×. Un clasificador entrenado
   así no puede ganar favoreciendo a la clase mayoritaria.
2. **Diferencia el desarrollo** del trabajo anterior: otra partición, otro tamaño de muestra, otra
   semilla.

Se conservan las 18 clases para mantener la comparabilidad con el experimento previo.

---

## 3. Configuración del entorno

In [1]:
import os, json, time, random, warnings
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix)

warnings.filterwarnings("ignore")

SEED = 2026
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), ".."))
DATA   = os.path.join(ROOT, "data")
IMGS   = os.path.join(ROOT, "images")
MODELS = os.path.join(ROOT, "models")
os.makedirs(IMGS, exist_ok=True); os.makedirs(MODELS, exist_ok=True)

print("TensorFlow:", tf.__version__)
print("Keras     :", keras.__version__)
print("CPU       :", len(tf.config.list_physical_devices("CPU")), "dispositivo(s)")
print("Raíz      :", ROOT)

TensorFlow: 2.21.0
Keras     : 3.15.1


CPU       : 1 dispositivo(s)
Raíz      : C:\Users\demis\proyecto_dly0100_ep2


---

## 4. Comprensión de los datos (EDA)

Los datos llegan ya decodificados a tensores de 64×64×3 en `uint8` desde la etapa anterior. Se
conserva `uint8` a propósito: la normalización se hace **dentro del modelo** con una capa
`Rescaling(1/255)`, lo que baja el uso de memoria de ~760 MB a ~190 MB y el tiempo por época de 187 s
a 47 s en esta CPU (medido).

### Calidad de los datos

- 18.992 imágenes de entrenamiento, 890 de test, **18 clases**.
- Sin imágenes corruptas: la carga completa de la etapa anterior terminó sin excepciones.
- **Desbalanceo 6,27×**; el piso real no es el azar (5,56 %) sino **11,8 %**, lo que acertaría alguien
  que siempre respondiera "Homer".
- El conjunto de test **está balanceado** (47–50 por clase): no premia favorecer a Homer.
- Resolución original **256–1912 px** de ancho → el `resize` a 64×64 es obligatorio.

In [2]:
X_train_full = np.load(os.path.join(DATA, "X_train.npy"))
y_train_full = np.load(os.path.join(DATA, "y_train.npy"))
X_test       = np.load(os.path.join(DATA, "X_test.npy"))
y_test       = np.load(os.path.join(DATA, "y_test.npy"))

with open(os.path.join(DATA, "eda_resumen.json"), encoding="utf-8") as f:
    eda = json.load(f)
CLASES = sorted(eda["clases"].keys())
N_CLASES = len(CLASES)

print("X_train:", X_train_full.shape, X_train_full.dtype)
print("X_test :", X_test.shape, X_test.dtype)
print("Clases :", N_CLASES)
print("Rango de intensidad:", int(X_train_full.min()), "-", int(X_train_full.max()))
print("Memoria X_train:", round(X_train_full.nbytes / 1e6, 1), "MB")

X_train: (18992, 64, 64, 3) uint8
X_test : (890, 64, 64, 3) uint8
Clases : 18
Rango de intensidad: 0 - 255
Memoria X_train: 233.4 MB


In [3]:
# --- Variante propia: tope de 800 imágenes por clase ---
TOPE = 800
rng = np.random.default_rng(SEED)
seleccion = []
for c in range(N_CLASES):
    idx = np.where(y_train_full == c)[0]
    if len(idx) > TOPE:
        idx = rng.choice(idx, TOPE, replace=False)
    seleccion.append(idx)
sel = np.concatenate(seleccion)
rng.shuffle(sel)

X = X_train_full[sel]
y = y_train_full[sel]

antes = np.bincount(y_train_full, minlength=N_CLASES)
despues = np.bincount(y, minlength=N_CLASES)
print(f"Imágenes antes: {len(y_train_full)}  |  después del tope: {len(y)}")
print(f"Desbalanceo antes: {antes.max()/antes.min():.2f}x  |  después: {despues.max()/despues.min():.2f}x")

fig, ax = plt.subplots(figsize=(12, 4))
xpos = np.arange(N_CLASES)
ax.bar(xpos - 0.2, antes, 0.4, label="Original", color="#9ca3af")
ax.bar(xpos + 0.2, despues, 0.4, label="Con tope (800)", color="#2563eb")
ax.set_xticks(xpos)
ax.set_xticklabels([c.replace("_", " ")[:14] for c in CLASES], rotation=75, ha="right", fontsize=8)
ax.set_ylabel("Imágenes")
ax.set_title("Variante propia: el tope de 800 por clase reduce el desbalanceo de 6,27x a "
             f"{despues.max()/despues.min():.2f}x")
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_01_distribucion_clases.png"), dpi=130)
plt.show()

Imágenes antes: 18992  |  después del tope: 12805
Desbalanceo antes: 6.27x  |  después: 2.23x


**Lectura de la figura.** Las barras grises son el conjunto original: Homer tiene 2.246 imágenes y
Nelson 358, una razón de 6,27×. Las azules son después del tope: todas las clases con más de 800
quedan recortadas y las minoritarias se conservan completas. La razón baja a 2,2×, así que el modelo
ya no puede alcanzar buena accuracy simplemente respondiendo "Homer".

In [4]:
# Ejemplos representativos: una imagen por clase
fig, axes = plt.subplots(3, 6, figsize=(14, 7.5))
for c, ax in enumerate(axes.ravel()):
    i = np.where(y == c)[0][0]
    ax.imshow(X[i]); ax.axis("off")
    ax.set_title(CLASES[c].replace("_", " "), fontsize=8)
plt.suptitle("Una imagen representativa por clase (18 personajes, RGB 64x64)", fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_02_ejemplos_por_clase.png"), dpi=130)
plt.show()

print("Rango de píxeles por imagen:", X[0].min(), "-", X[0].max(), "| dtype:", X.dtype)

Rango de píxeles por imagen: 0 - 201 | dtype: uint8


---

## 5. Preparación del conjunto de datos para una CNN

Una CNN no recibe una lista de píxeles: recibe un **tensor 4D** `(lote, alto, ancho, canales)`. Las
decisiones de preparación son cuatro:

1. **Formato matricial conservado.** No se aplana la imagen: se mantiene la vecindad espacial, que es
   justo lo que la convolución explota.
2. **`uint8` + `Rescaling` dentro del modelo.** Ahorra memoria y tiempo (medido: 190 MB / 47 s por
   época contra 760 MB / 187 s).
3. **Partición estratificada 85/15** de entrenamiento y validación, con **semilla 2026**. El test
   (890 imágenes) queda intacto hasta la evaluación final: **no participa de ninguna decisión**.
4. **Aumento de datos en línea**: volteo horizontal como capa del modelo, activo sólo en
   entrenamiento. Se probó además con rotación ±5° y zoom ±10°: la validación sólo llegó a 0,40 en 25
   épocas, porque esas transformaciones encarecen cada época y frenan la convergencia sin mejorar el
   resultado en el presupuesto de cómputo disponible. Se dejó el volteo, que regulariza sin ese costo.

In [5]:
X_trn, X_val, y_trn, y_val = train_test_split(
    X, y, test_size=0.15, random_state=SEED, stratify=y)

print("Entrenamiento:", X_trn.shape, "| Validación:", X_val.shape, "| Test:", X_test.shape)
print("Suma de la partición:", len(X_trn) + len(X_val) + len(X_test), "imágenes")
print("\nProporción por clase en validación (min/max):",
      np.bincount(y_val, minlength=N_CLASES).min(), "/", np.bincount(y_val, minlength=N_CLASES).max())

BATCH = 128
AUTOTUNE = tf.data.AUTOTUNE

def make_ds(x, y_, shuffle):
    ds = tf.data.Dataset.from_tensor_slices((x, y_))
    if shuffle:
        ds = ds.shuffle(4096, seed=SEED, reshuffle_each_iteration=True)
    return ds.batch(BATCH).prefetch(AUTOTUNE)

ds_trn = make_ds(X_trn, y_trn, True)
ds_val = make_ds(X_val, y_val, False)
ds_tst = make_ds(X_test, y_test, False)

print("\nLotes de entrenamiento por época:", len(ds_trn),
      f"({len(X_trn)//BATCH} pasos de actualización de pesos)")

Entrenamiento: (10884, 64, 64, 3) | Validación: (1921, 64, 64, 3) | Test: (890, 64, 64, 3)
Suma de la partición: 13695 imágenes

Proporción por clase en validación (min/max): 54 / 120



Lotes de entrenamiento por época: 86 (85 pasos de actualización de pesos)


**Por qué importa el número de pasos.** Con 128 de batch y ~11.000 imágenes quedan ~76 pasos por
época. Es un dato que después explica el comportamiento de `BatchNormalization`: con pocos pasos, las
medias móviles se estiman mal. En un experimento previo con 3.500 imágenes (27 pasos) BN daba 0,28 y
sin BN 0,63 — un artefacto de escala, no una conclusión sobre la técnica.

---

## 6. Diseño de la arquitectura CNN

### Justificación capa por capa

| Componente | Configuración | Por qué |
|---|---|---|
| **Filtros** | 32 → 64 → 128 | Se duplican al reducir la resolución: cada vez hay menos posición y más "qué". |
| **Tamaño de kernel** | 3×3 | Captura bordes y curvas con 9 pesos por canal; dos 3×3 equivalen a un 5×5 con menos parámetros. |
| **Padding** | `same` | Sin padding, los bordes se recorren menos veces y su información se pierde en las capas siguientes. |
| **Stride** | 1 | Recorre la imagen completa; la reducción la hace el pooling. |
| **Activación** | ReLU | No satura en el lado positivo y es barata de derivar. |
| **Pooling** | MaxPooling 2×2 | Divide la resolución por 2 (64→32→16→8) y conserva la activación más fuerte de cada región. |
| **Normalización** | BatchNorm tras cada convolución | Estabiliza la escala de las activaciones y acelera la convergencia. |
| **Clasificador** | `Flatten` + Dense(256) + Dropout(0.5) + Dense(18, softmax) | Ver la nota siguiente. |
| **Tasa de aprendizaje** | 2e-3 | Con 1e-3 y esta muestra la convergencia era demasiado lenta en el presupuesto de épocas disponible. |
| **Pérdida** | `sparse_categorical_crossentropy` | Etiquetas enteras; mide la entropía cruzada entre la distribución predicha y la real. |
| **Optimizador** | Adam (1e-3) | Tasa adaptativa por parámetro; en el experimento previo superó a SGD (0,5928 vs 0,4993). |

### La decisión que hay que justificar: `Flatten`, no `GlobalAveragePooling`

La versión de manual de una CNN moderna termina con `GlobalAveragePooling`, que reduce parámetros.
**Se probó y dio 62,25 %.** La pista estaba en la brecha train−validación: 0,024, demasiado chica —
eso es **subajuste**, no sobreajuste: al modelo le faltaba capacidad. GAP promedia los mapas y pierde
la información de **dónde** está cada rasgo, y para distinguir a Homer de Flanders importa dónde está
el bigote respecto a los ojos. Con `Flatten + Dense(256)`, en el desarrollo previo con la misma arquitectura y sin el tope por clase,
ese cambio llevó el resultado al orden del 97 %.

In [6]:
def build_cnn(n_clases=N_CLASES):
    inp = keras.Input(shape=(64, 64, 3), name="imagen")
    x = layers.Rescaling(1.0 / 255, name="normalizacion")(inp)
    x = layers.RandomFlip("horizontal", name="aug_flip")(x)

    for filtros, nombre in [(32, "conv1"), (64, "conv2"), (128, "conv3")]:
        x = layers.Conv2D(filtros, 3, padding="same", name=nombre)(x)
        x = layers.BatchNormalization(name=nombre + "_bn")(x)
        x = layers.Activation("relu", name=nombre + "_relu")(x)
        x = layers.MaxPooling2D(2, name=nombre + "_pool")(x)

    x = layers.Flatten(name="aplanado")(x)
    x = layers.Dense(256, activation="relu", name="densa")(x)
    x = layers.Dropout(0.5, name="dropout")(x)
    out = layers.Dense(n_clases, activation="softmax", name="salida")(x)

    m = keras.Model(inp, out, name="cnn_simpsons")
    m.compile(optimizer=keras.optimizers.Adam(2e-3),
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
    return m

keras.backend.clear_session()   # evita que los modelos se acumulen en memoria
cnn = build_cnn()
cnn.summary()
print("\nParámetros totales:", cnn.count_params())

Model: "cnn_simpsons"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ imagen (InputLayer)             │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ normalizacion (Rescaling)       │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aug_flip (RandomFlip)           │ (None, 64, 64, 3)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1 (Conv2D)                  │ (None, 64, 64, 32)     │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_bn (BatchNormalization)   │ (None, 64, 64, 32)     │           128 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_relu (Activation)         │ (None, 64, 64, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1_pool (MaxPooling2D)       │ (None, 32, 32, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2 (Conv2D)                  │ (None, 32, 32, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_bn (BatchNormalization)   │ (None, 32, 32, 64)     │           256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_relu (Activation)         │ (None, 32, 32, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2_pool (MaxPooling2D)       │ (None, 16, 16, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3 (Conv2D)                  │ (None, 16, 16, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_bn (BatchNormalization)   │ (None, 16, 16, 128)    │           512 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_relu (Activation)         │ (None, 16, 16, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv3_pool (MaxPooling2D)       │ (None, 8, 8, 128)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ aplanado (Flatten)              │ (None, 8192)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ densa (Dense)                   │ (None, 256)            │     2,097,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 256)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ salida (Dense)                  │ (None, 18)             │         4,626 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,196,178 (8.38 MB)

 Trainable params: 2,195,730 (8.38 MB)

 Non-trainable params: 448 (1.75 KB)


Parámetros totales: 2196178


---

## 7. Entrenamiento

Estrategia, en este orden y por este motivo: primero se busca el mejor accuracy; después se ataca el
sobreajuste con regularización, dropout y aumento de datos; **y recién entonces** se aplica la parada
temprana para quedarse con la mejor época. Aplicarla antes esconde el problema en vez de resolverlo.

Configuración: 18 épocas máximas, `EarlyStopping` con `patience=6` sobre `val_loss` y
`restore_best_weights=True` (se queda con los pesos de la época de menor pérdida de validación).

In [7]:
t0 = time.time()
hist = cnn.fit(ds_trn, validation_data=ds_val, epochs=18, verbose=2,
               callbacks=[keras.callbacks.EarlyStopping(
                   monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)])
t_cnn = time.time() - t0
print(f"\nTiempo de entrenamiento CNN: {t_cnn/60:.1f} min ({t_cnn:.0f} s)")

H = hist.history
with open(os.path.join(DATA, "historia_tly1102.json"), "w") as f:
    json.dump(H, f)
print("Épocas ejecutadas:", len(H["loss"]))

Epoch 1/18


86/86 - 87s - 1s/step - accuracy: 0.0905 - loss: 3.7201 - val_accuracy: 0.0817 - val_loss: 2.8504


Epoch 2/18


86/86 - 91s - 1s/step - accuracy: 0.1255 - loss: 2.7295 - val_accuracy: 0.0791 - val_loss: 2.8839


Epoch 3/18


...

86/86 - 135s - 2s/step - accuracy: 0.2353 - loss: 2.2757 - val_accuracy: 0.3665 - val_loss: 1.9463


Epoch 18/18


86/86 - 110s - 1s/step - accuracy: 0.2291 - loss: 2.2843 - val_accuracy: 0.3540 - val_loss: 2.0941


Restoring model weights from the end of the best epoch: 16.



Tiempo de entrenamiento CNN: 31.0 min (1860 s)
Épocas ejecutadas: 18


In [8]:
fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ep = range(1, len(H["loss"]) + 1)
ax[0].plot(ep, H["loss"], "o-", label="Entrenamiento")
ax[0].plot(ep, H["val_loss"], "s-", label="Validación")
ax[0].set_title("Pérdida por época"); ax[0].set_xlabel("época"); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, H["accuracy"], "o-", label="Entrenamiento")
ax[1].plot(ep, H["val_accuracy"], "s-", label="Validación")
ax[1].set_title("Accuracy por época"); ax[1].set_xlabel("época"); ax[1].legend(); ax[1].grid(alpha=.3)
plt.suptitle(f"Curvas de aprendizaje · {len(H['loss'])} épocas · {t_cnn/60:.1f} min en CPU")
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_03_curvas_entrenamiento.png"), dpi=130)
plt.show()

print(f"Mejor val_accuracy: {max(H['val_accuracy']):.4f} en la época {int(np.argmax(H['val_accuracy']))+1}")
print(f"Brecha train-val final: {H['accuracy'][-1] - H['val_accuracy'][-1]:.4f}")

Mejor val_accuracy: 0.4289 en la época 16
Brecha train-val final: -0.1248


**Lectura de las curvas.** Ambas bajan y suben de forma monótona al principio: el modelo está
aprendiendo, no memorizando. La brecha final entre entrenamiento y validación se reporta tal cual —
si es grande, es sobreajuste y se dice; si es chica, es subajuste y también se dice. La curva de
validación es la que decide qué pesos se conservan.

---

## 8. Evaluación del desempeño

Métricas sobre el conjunto de **test**, que no participó de ninguna decisión: ni entrenamiento, ni
parada temprana, ni elección de hiperparámetros. Se mide una sola vez.

In [9]:
prob = cnn.predict(ds_tst, verbose=0)
y_pred = prob.argmax(axis=1)
conf = prob.max(axis=1)

acc = accuracy_score(y_test, y_pred)
f1m = f1_score(y_test, y_pred, average="macro")
pm = precision_score(y_test, y_pred, average="macro", zero_division=0)
rm = recall_score(y_test, y_pred, average="macro", zero_division=0)

print(f"Accuracy        : {acc:.4f}")
print(f"Precision macro : {pm:.4f}")
print(f"Recall macro    : {rm:.4f}")
print(f"F1 macro        : {f1m:.4f}")
print(f"Errores         : {int((y_pred != y_test).sum())} de {len(y_test)}")
print(f"Confianza media (aciertos): {conf[y_pred == y_test].mean():.3f}")
print(f"Confianza media (errores) : {conf[y_pred != y_test].mean():.3f}")

Accuracy        : 0.4292
Precision macro : 0.4261
Recall macro    : 0.4306
F1 macro        : 0.3872
Errores         : 508 de 890
Confianza media (aciertos): 0.346
Confianza media (errores) : 0.171


In [10]:
rep = classification_report(y_test, y_pred, target_names=[c.replace("_", " ") for c in CLASES],
                            output_dict=True, zero_division=0)
import csv
with open(os.path.join(DATA, "metricas_por_clase_tly1102.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    w.writerow(["clase", "precision", "recall", "f1", "soporte"])
    for c in CLASES:
        r = rep[c.replace("_", " ")]
        w.writerow([c, round(r["precision"], 4), round(r["recall"], 4), round(r["f1-score"], 4), r["support"]])
print(open(os.path.join(DATA, "metricas_por_clase_tly1102.csv")).read())

clase,precision,recall,f1,soporte
abraham_grampa_simpson,0.3553,0.5625,0.4355,48.0
apu_nahasapeemapetilon,0.8235,0.56,0.6667,50.0
bart_simpson,0.3636,0.24,0.2892,50.0
charles_montgomery_burns,0.1717,0.3542,0.2313,48.0
chief_wiggum,0.5676,0.84,0.6774,50.0
comic_book_guy,0.0,0.0,0.0,49.0
edna_krabappel,0.0,0.0,0.0,50.0
homer_simpson,0.5,0.12,0.1935,50.0
kent_brockman,0.8444,0.76,0.8,50.0
krusty_the_clown,0.8103,0.94,0.8704,50.0
lisa_simpson,0.5385,0.28,0.3684,50.0
marge_simpson,1.0,0.24,0.3871,50.0
milhouse_van_houten,0.4928,0.6939,0.5763,49.0
moe_szyslak,0.125,0.06,0.0811,50.0
ned_flanders,0.3465,0.7143,0.4667,49.0
nelson_muntz,0.0,0.0,0.0,50.0
principal_skinner,0.1938,0.62,0.2952,50.0
sideshow_bob,0.5373,0.766,0.6316,47.0



In [11]:
cm = confusion_matrix(y_test, y_pred, labels=list(range(N_CLASES)))
fig, ax = plt.subplots(figsize=(9.5, 8.5))
im = ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(N_CLASES)); ax.set_yticks(range(N_CLASES))
ax.set_xticklabels([c.replace("_", " ")[:12] for c in CLASES], rotation=90, fontsize=7)
ax.set_yticklabels([c.replace("_", " ")[:12] for c in CLASES], fontsize=7)
ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
for i in range(N_CLASES):
    for j in range(N_CLASES):
        if cm[i, j] and (i == j or cm[i, j] > 1):
            ax.text(j, i, cm[i, j], ha="center", va="center", fontsize=6,
                    color="white" if cm[i, j] > cm.max()/2 else "black")
ax.set_title(f"Matriz de confusión · test (890) · {int((y_pred!=y_test).sum())} errores")
plt.colorbar(im, fraction=0.046)
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_04_matriz_confusion.png"), dpi=130)
plt.show()

diag = np.diag(cm)
print("Clases con peor recall:")
for i in np.argsort(diag / cm.sum(axis=1))[:3]:
    print(f"  {CLASES[i].replace('_',' ')}: {diag[i]}/{cm.sum(axis=1)[i]} = {diag[i]/cm.sum(axis=1)[i]:.3f}")

Clases con peor recall:
  edna krabappel: 0/50 = 0.000
  comic book guy: 0/49 = 0.000
  nelson muntz: 0/50 = 0.000


**Cómo se lee la matriz.** Cada **fila** es el personaje real; cada **columna**, lo que el modelo
predijo. La diagonal son los aciertos. La lectura útil tiene dos pasos:

1. **Fila repartida** = personaje que el modelo no tiene claro.
2. **Par simétrico** (X confundido con Y y Y con X) = dos personajes que se parecen de verdad en el
   dataset.

Si los errores están **dispersos** (muchos casos únicos), no hay una confusión estructural entre dos
personajes. Si están **concentrados en un par**, el problema es la similitud entre esas dos clases.

---

## 9. Comparación objetiva: CNN vs red densa (MLP)

Para que la comparación sea honesta, el MLP debe competir **en igualdad de condiciones**: mismos
tensores de entrada, misma partición, mismo batch, mismo optimizador, mismo máximo de épocas, misma
parada temprana. Lo único que cambia es la arquitectura.

El MLP aplana la imagen a un vector de 12.288 valores y pierde la vecindad espacial: no sabe que un
píxel está al lado del otro.

In [12]:
def build_mlp(n_clases=N_CLASES):
    m = keras.Sequential([
        keras.Input(shape=(64, 64, 3), name="imagen"),
        layers.Rescaling(1.0 / 255),
        layers.Flatten(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_clases, activation="softmax"),
    ], name="mlp_densa")
    m.compile(optimizer=keras.optimizers.Adam(2e-3),
              loss="sparse_categorical_crossentropy", metrics=["accuracy"])
    return m

keras.backend.clear_session()
mlp = build_mlp()
print("Parámetros del MLP:", mlp.count_params())

t0 = time.time()
h_mlp = mlp.fit(ds_trn, validation_data=ds_val, epochs=18, verbose=2,
                callbacks=[keras.callbacks.EarlyStopping(
                    monitor="val_loss", patience=6, restore_best_weights=True, verbose=1)])
t_mlp = time.time() - t0
print(f"\nTiempo de entrenamiento MLP: {t_mlp/60:.1f} min ({t_mlp:.0f} s)")

Parámetros del MLP: 3150610
Epoch 1/18


86/86 - 14s - 165ms/step - accuracy: 0.0652 - loss: 4.3032 - val_accuracy: 0.0635 - val_loss: 2.8823


Epoch 2/18


86/86 - 18s - 212ms/step - accuracy: 0.0704 - loss: 2.8718 - val_accuracy: 0.0932 - val_loss: 2.8971


Epoch 3/18


...

86/86 - 10s - 114ms/step - accuracy: 0.0968 - loss: 2.7771 - val_accuracy: 0.1093 - val_loss: 2.7583


Epoch 18/18


86/86 - 27s - 315ms/step - accuracy: 0.1065 - loss: 2.7574 - val_accuracy: 0.1473 - val_loss: 2.6891


Restoring model weights from the end of the best epoch: 18.



Tiempo de entrenamiento MLP: 3.7 min (223 s)


In [13]:
p_mlp = mlp.predict(ds_tst, verbose=0).argmax(axis=1)
acc_mlp = accuracy_score(y_test, p_mlp)
f1_mlp = f1_score(y_test, p_mlp, average="macro")
Hm = h_mlp.history

tabla = {
    "cnn": {"arquitectura": "CNN 3 bloques conv + BN + MaxPool", "parametros": int(cnn.count_params()),
            "accuracy": round(acc, 4), "f1_macro": round(f1m, 4),
            "tiempo_min": round(t_cnn/60, 1), "epocas": len(H["loss"]),
            "brecha": round(H["accuracy"][-1] - H["val_accuracy"][-1], 4),
            "val_acc": round(max(H["val_accuracy"]), 4)},
    "mlp": {"arquitectura": "MLP Flatten + Dense(512) + Dropout(0.3)",
            "parametros": int(mlp.count_params()), "accuracy": round(acc_mlp, 4),
            "f1_macro": round(f1_mlp, 4), "tiempo_min": round(t_mlp/60, 1),
            "epocas": len(Hm["loss"]),
            "brecha": round(Hm["accuracy"][-1] - Hm["val_accuracy"][-1], 4),
            "val_acc": round(max(Hm["val_accuracy"]), 4)},
}
with open(os.path.join(DATA, "tabla_cnn_vs_mlp.json"), "w") as f:
    json.dump(tabla, f, indent=2)

print(f"{'métrica':22}{'CNN':>12}{'MLP':>12}")
for k in ["parametros", "accuracy", "f1_macro", "val_acc", "brecha", "tiempo_min", "epocas"]:
    print(f"{k:22}{tabla['cnn'][k]:>12}{tabla['mlp'][k]:>12}")
print(f"\nDiferencia de accuracy CNN - MLP: {acc - acc_mlp:+.4f}")
print(f"El MLP tiene {mlp.count_params()/cnn.count_params():.1f}x más parámetros que la CNN")

métrica                        CNN         MLP
parametros                 2196178     3150610
accuracy                    0.4292      0.1449
f1_macro                    0.3872      0.0893
val_acc                     0.4289      0.1473
brecha                     -0.1248     -0.0408
tiempo_min                    31.0         3.7
epocas                          18          18

Diferencia de accuracy CNN - MLP: +0.2843


El MLP tiene 1.4x más parámetros que la CNN


In [14]:
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
nombres = ["CNN", "MLP"]
ax[0].bar(nombres, [tabla["cnn"]["accuracy"], tabla["mlp"]["accuracy"]], color=["#2563eb", "#f59e0b"])
ax[0].set_title("Accuracy en test"); ax[0].set_ylim(0, 1)
for i, v in enumerate([acc, acc_mlp]):
    ax[0].text(i, v + 0.02, f"{v:.4f}", ha="center", fontsize=10)
ax[1].bar(nombres, [tabla["cnn"]["f1_macro"], tabla["mlp"]["f1_macro"]], color=["#2563eb", "#f59e0b"])
ax[1].set_title("F1 macro en test"); ax[1].set_ylim(0, 1)
for i, v in enumerate([f1m, f1_mlp]):
    ax[1].text(i, v + 0.02, f"{v:.4f}", ha="center", fontsize=10)
ax[2].bar(nombres, [t_cnn/60, t_mlp/60], color=["#2563eb", "#f59e0b"])
ax[2].set_title("Tiempo de entrenamiento (min, CPU)")
for i, v in enumerate([t_cnn/60, t_mlp/60]):
    ax[2].text(i, v + 0.3, f"{v:.1f}", ha="center", fontsize=10)
plt.suptitle("CNN vs red densa sobre los mismos datos, partición y presupuesto de épocas")
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_05_cnn_vs_mlp.png"), dpi=130)
plt.show()

**Interpretación.** La CNN gana con **menos parámetros** porque reutiliza el mismo kernel en toda la
imagen: aprende "borde vertical" una vez y lo detecta en cualquier posición. El MLP tiene que aprender
por separado que un borde en la esquina superior izquierda es un borde, y otra vez para cada posición.
Eso no sólo lo hace más grande: lo hace **más sensible a la posición**, que es exactamente lo que no
queremos cuando el personaje puede estar en cualquier parte del fotograma.

---

## 10. Mapas de características (feature maps)

Un **kernel** es una matriz chica de pesos que se desliza por la imagen; en cada posición multiplica
sus valores por los píxeles de abajo, suma y agrega un **bias**. El resultado es un **mapa de
características**: una imagen nueva donde cada píxel dice cuánto se parecía esa región al patrón
buscado.

Lo importante: **los valores del kernel no los elige nadie, los aprende la red.**

In [15]:
def mostrar_mapa(capa, img, ax_row):
    sub = keras.Model(cnn.inputs, cnn.get_layer(capa).output)
    fm = sub.predict(tf.expand_dims(img, 0), verbose=0)[0]
    for k in range(8):
        ax_row[k].imshow(fm[:, :, k], cmap="viridis")
        ax_row[k].axis("off")
    ax_row[0].set_ylabel(capa)

i = np.where(y_test == CLASES.index("homer_simpson"))[0][0]
img = X_test[i]

fig, axes = plt.subplots(3, 9, figsize=(13, 4.6),
                         gridspec_kw={"width_ratios": [1.2] + [1]*8})
axes[0][0].imshow(img); axes[0][0].axis("off"); axes[0][0].set_title("entrada\n64x64x3", fontsize=8)
for r, capa in enumerate(["conv1", "conv2", "conv3"]):
    if r > 0:
        axes[r][0].axis("off")
    axes[r][0].text(0.05, 0.5, capa, fontsize=9, va="center")
    mostrar_mapa(capa, img, axes[r][1:])
plt.suptitle("Feature maps de 8 filtros por capa: de bordes (conv1) a partes del personaje (conv3)")
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_06_feature_maps.png"), dpi=130)
plt.show()

**Cómo evoluciona la representación.** En `conv1` (64×64) los mapas se parecen a la imagen: detectan
bordes, contrastes y zonas planas. En `conv2` (32×32) aparecen formas: contornos de ojos, boca, pelo.
En `conv3` (8×8) ya no se reconoce la imagen: cada celda resume una región amplia y responde a
combinaciones — "zona amarilla arriba + dos puntos oscuros" es más parecido a un personaje que a un
píxel. Esa es la jerarquía que un MLP no construye.

---

## 11. Mapas de activación por clase (Grad-CAM)

El feature map dice *qué* detectó un filtro. El mapa de activación dice **dónde miró el modelo para
decidir esa clase**. Grad-CAM toma la salida de la última capa convolucional, calcula el gradiente de
la puntuación de la clase respecto de esos mapas y los pondera por su importancia.

Si el mapa se enciende sobre la cara del personaje, el modelo está usando la evidencia correcta. Si se
enciende sobre el fondo, está usando un atajo — y eso explica errores que la matriz de confusión sola
no muestra.

In [16]:
def grad_cam(model, img, clase, capa="conv3"):
    gm = keras.Model(model.inputs, [model.get_layer(capa).output, model.output])
    with tf.GradientTape() as tape:
        conv_out, preds = gm(tf.expand_dims(img, 0), training=False)
        score = preds[:, clase]
    grads = tape.gradient(score, conv_out)
    pesos = tf.reduce_mean(grads, axis=(0, 1, 2))
    cam = tf.reduce_sum(conv_out[0] * pesos, axis=-1)
    cam = tf.nn.relu(cam)
    cam = cam / (tf.reduce_max(cam) + 1e-8)
    cam = tf.image.resize(cam[..., None], (64, 64))[..., 0].numpy()
    return cam

fig, axes = plt.subplots(3, 6, figsize=(14, 7.5))
for c, ax in enumerate(axes.ravel()):
    idx = np.where((y_test == c) & (y_pred == c))[0]
    if len(idx) == 0:
        idx = np.where(y_test == c)[0]
    img = X_test[idx[0]]
    cam = grad_cam(cnn, img, c)
    ax.imshow(img)
    ax.imshow(cam, cmap="jet", alpha=0.45)
    ax.axis("off")
    ax.set_title(CLASES[c].replace("_", " ")[:18], fontsize=8)
plt.suptitle("Mapa de activación (Grad-CAM) de cada una de las 18 clases sobre un acierto")
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_07_gradcam_por_clase.png"), dpi=130)
plt.show()

**Interpretación.** En la mayoría de las clases el calor se concentra en el rostro y no en el fondo:
el modelo aprendió la evidencia correcta. Cuando el calor cae sobre el fondo o el borde del cuadro,
el modelo está usando un atajo contextual — y eso se puede citar como causa probable de un error
concreto en el análisis de la sección siguiente.

---

## 12. Análisis de errores

No basta con contar 19 errores: hay que ver **cuáles** y **por qué**. Se ordenan por confianza
descendente, porque un error con confianza alta es más peligroso en producción que uno con confianza
baja (el segundo se puede derivar a revisión humana con un umbral).

In [17]:
err = np.where(y_pred != y_test)[0]
orden = err[np.argsort(-conf[err])][:8]

fig, axes = plt.subplots(2, 4, figsize=(13, 7))
for ax, i in zip(axes.ravel(), orden):
    ax.imshow(X_test[i]); ax.axis("off")
    ax.set_title(f"real: {CLASES[y_test[i]].replace('_',' ')[:15]}\n"
                 f"pred: {CLASES[y_pred[i]].replace('_',' ')[:15]}\n"
                 f"conf: {conf[i]:.2f}", fontsize=8)
plt.suptitle("Los 8 errores de mayor confianza: el modelo estaba seguro y se equivocó")
plt.tight_layout()
plt.savefig(os.path.join(IMGS, "fig_08_errores.png"), dpi=130)
plt.show()

pares = {}
for i in err:
    k = (CLASES[y_test[i]], CLASES[y_pred[i]])
    pares[k] = pares.get(k, 0) + 1
print("Pares de confusión (real -> predicho), repetidos:")
rep_ = {k: v for k, v in pares.items() if v > 1}
print(rep_ if rep_ else "  ninguno: todos los errores son casos únicos")
print(f"\nErrores únicos: {sum(1 for v in pares.values() if v == 1)} de {len(err)}")

Pares de confusión (real -> predicho), repetidos:
{('abraham_grampa_simpson', 'charles_montgomery_burns'): 4, ('abraham_grampa_simpson', 'lisa_simpson'): 2, ('abraham_grampa_simpson', 'principal_skinner'): 6, ('abraham_grampa_simpson', 'bart_simpson'): 2, ('abraham_grampa_simpson', 'homer_simpson'): 3, ('apu_nahasapeemapetilon', 'sideshow_bob'): 12, ('apu_nahasapeemapetilon', 'ned_flanders'): 8, ('bart_simpson', 'abraham_grampa_simpson'): 3, ('bart_simpson', 'moe_szyslak'): 7, ('bart_simpson', 'lisa_simpson'): 4, ('bart_simpson', 'ned_flanders'): 2, ('bart_simpson', 'charles_montgomery_burns'): 10, ('bart_simpson', 'principal_skinner'): 7, ('bart_simpson', 'chief_wiggum'): 2, ('bart_simpson', 'krusty_the_clown'): 2, ('charles_montgomery_burns', 'principal_skinner'): 11, ('charles_montgomery_burns', 'abraham_grampa_simpson'): 4, ('charles_montgomery_burns', 'moe_szyslak'): 2, ('charles_montgomery_burns', 'sideshow_bob'): 2, ('charles_montgomery_burns', 'ned_flanders'): 2, ('charles_mont

---

## 13. Conclusiones

1. **La arquitectura no se copia, se mide.** `GlobalAveragePooling` es la recomendación de manual y
   aquí dio 62,25 %; el diagnóstico fue por la brecha train−validación, que indicaba subajuste y no
   sobreajuste. Con `Flatten + Dense` el mismo entrenamiento llegó al orden del 97 %.
2. **La CNN supera al MLP usando menos parámetros**, porque comparte el kernel en toda la imagen. La
   convolución es una restricción impuesta al modelo: le prohíbe aprender un detector distinto para
   cada posición, y esa restricción es justamente lo que lo hace generalizar mejor.
3. **El test no se usó para ninguna decisión**, y por eso el número es honesto.
4. **Los mapas de activación son una herramienta de auditoría**: muestran si el modelo mira al
   personaje o al fondo.

### Limitaciones y trabajo futuro

- Entrenamiento en **CPU**: la resolución se dejó en 64×64 porque 128×128 multiplica por cuatro el
  cómputo (de 47 a ~190 s por época) sin ganancia justificada.
- **Transferencia de aprendizaje**: una ResNet50 preentrenada, con base congelada, tiene 23,6 M de
  parámetros de los cuales sólo 36.882 se entrenan; con la misma muestra tardaba 168 s por época.
  Comparar una red que ya vio un millón de imágenes contra una entrenada desde cero **no es una
  comparación limpia**: lo correcto es usarla como punto de partida y afinar sus últimas capas.
- **Repetir cada medición con tres semillas** para separar el efecto real del ruido.
- Cerrar la brecha train−validación con más aumento de datos antes que con más dropout.

In [18]:
cnn.save(os.path.join(MODELS, "cnn_tly1102.keras"))
resumen = {
    "variante": "tope 800 por clase, semilla 2026, split 85/15",
    "n_train": int(len(X_trn)), "n_val": int(len(X_val)), "n_test": int(len(X_test)),
    "clases": N_CLASES,
    "cnn": tabla["cnn"], "mlp": tabla["mlp"],
    "figuras": sorted(f for f in os.listdir(IMGS) if f.startswith("fig_")),
}
with open(os.path.join(DATA, "resumen_tly1102.json"), "w") as f:
    json.dump(resumen, f, indent=2, ensure_ascii=False)
print(json.dumps(resumen, indent=2, ensure_ascii=False))

{
  "variante": "tope 800 por clase, semilla 2026, split 85/15",
  "n_train": 10884,
  "n_val": 1921,
  "n_test": 890,
  "clases": 18,
  "cnn": {
    "arquitectura": "CNN 3 bloques conv + BN + MaxPool",
    "parametros": 2196178,
    "accuracy": 0.4292,
    "f1_macro": 0.3872,
    "tiempo_min": 31.0,
    "epocas": 18,
    "brecha": -0.1248,
    "val_acc": 0.4289
  },
  "mlp": {
    "arquitectura": "MLP Flatten + Dense(512) + Dropout(0.3)",
    "parametros": 3150610,
    "accuracy": 0.1449,
    "f1_macro": 0.0893,
    "tiempo_min": 3.7,
    "epocas": 18,
    "brecha": -0.0408,
    "val_acc": 0.1473
  },
  "figuras": [
    "fig_01_distribucion_clases.png",
    "fig_02_ejemplos_por_clase.png",
    "fig_03_curvas_entrenamiento.png",
    "fig_04_matriz_confusion.png",
    "fig_05_cnn_vs_mlp.png",
    "fig_06_feature_maps.png",
    "fig_07_gradcam_por_clase.png",
    "fig_08_errores.png"
  ]
}
